In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, f1_score
from transformers import BertTokenizer, BertForSequenceClassification, Trainer, TrainingArguments
import torch
from torch.utils.data import Dataset

In [ ]:
# Load the preprocessed dataset
df = pd.read_csv('../DATASETS/SA_preprocessed.csv')
print(df.head())

# Check for missing values
print(df.isnull().sum())

# Use 'combined' as input text and 'target' as label
texts = df['processed_text'].tolist()
labels = df['label'].tolist()

   label                                               data  \
0      0  ["Subject: [zzzzteana] RE: Alexander  \nMartin...   
1      0  ["Subject: [zzzzteana] Moscow bomber  \nMan Th...   
2      0  ["Subject: [IRR] Klez: The Virus That  Won't D...   
3      0  ["Subject: Re: [zzzzteana] Nothing like mama u...   
4      0  ["Subject: Re: [zzzzteana] Nothing like mama u...   

                                 filename  \
0  00002.9c4069e25e1ef370c078db7ee85ff9ac   
1  00003.860e3c3cee1b42ead714c5c874fe25f7   
2  00004.864220c5b6930b209cc287c361c99af1   
3  00005.bf27cdeaf0b8c4647ecd61b1d09da613   
4  00006.253ea2f9a9cc36fa0b1129b04b806608   

                                      processed_text  
0  zzzzteana re alexander martin a posted tassos ...  
1  zzzzteana moscow bomber man threatens explosio...  
2  irr klez the virus that will not die already t...  
3  re zzzzteana nothing like mama used to make in...  
4  re zzzzteana nothing like mama used to make i ...  
label             0


In [3]:
# Load BERT tokenizer
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

# Split the data
train_texts, val_texts, train_labels, val_labels = train_test_split(texts, labels, test_size=0.2, random_state=42)

# Filter out non-string texts and corresponding labels
# This is necessary because 'processed_text' column had 2 missing values,
# which become float('nan') in the list and cause TypeError with the tokenizer.
train_filtered_data = [(text, label) for text, label in zip(train_texts, train_labels) if isinstance(text, str)]
train_texts = [item[0] for item in train_filtered_data]
train_labels = [item[1] for item in train_filtered_data]

val_filtered_data = [(text, label) for text, label in zip(val_texts, val_labels) if isinstance(text, str)]
val_texts = [item[0] for item in val_filtered_data]
val_labels = [item[1] for item in val_filtered_data]


# Tokenize the texts
def tokenize_function(texts):
    return tokenizer(texts, padding=True, truncation=True, max_length=512, return_tensors='pt')

train_encodings = tokenize_function(train_texts)
val_encodings = tokenize_function(val_texts)

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [4]:
# Custom Dataset class
class SpamDataset(Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {key: val[idx] for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)

train_dataset = SpamDataset(train_encodings, train_labels)
val_dataset = SpamDataset(val_encodings, val_labels)

In [5]:
# Load pre-trained BERT model for sequence classification
model = BertForSequenceClassification.from_pretrained('bert-base-uncased', num_labels=2)

# Training arguments
training_args = TrainingArguments(
    output_dir='./results',
    num_train_epochs=3,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    warmup_steps=500,
    weight_decay=0.01,
    logging_dir='./logs',
    logging_steps=10,
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
)

# Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
)

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
`logging_dir` is deprecated and will 

In [6]:
# Train the model
trainer.train()

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Epoch,Training Loss,Validation Loss
1,0.055837,0.231526
2,0.066489,0.128614
3,0.000441,0.161563


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.attention.output.La

TrainOutput(global_step=909, training_loss=0.24325757290977407, metrics={'train_runtime': 851.1247, 'train_samples_per_second': 17.039, 'train_steps_per_second': 1.068, 'total_flos': 3815636524830720.0, 'train_loss': 0.24325757290977407, 'epoch': 3.0})

In [7]:
# Evaluate the model
predictions = trainer.predict(val_dataset)
preds = torch.argmax(torch.tensor(predictions.predictions), axis=1)

print('Accuracy:', accuracy_score(val_labels, preds))
print('F1 Score:', f1_score(val_labels, preds))
print('Classification Report:')
print(classification_report(val_labels, preds))

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Accuracy: 0.9851116625310173
F1 Score: 0.9752747252747253
Classification Report:
              precision    recall  f1-score   support

           0       0.99      0.99      0.99       842
           1       0.98      0.97      0.98       367

    accuracy                           0.99      1209
   macro avg       0.98      0.98      0.98      1209
weighted avg       0.99      0.99      0.99      1209



In [8]:
# Save the model and tokenizer to a directory
model_path = './spam_filter_bert'
trainer.save_model(model_path)
tokenizer.save_pretrained(model_path)

print(f'Model saved to {model_path}')

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model saved to ./spam_filter_bert


In [9]:
import torch
from transformers import BertTokenizer, BertForSequenceClassification

# Path where the model was saved
model_path = './spam_filter_bert/'

# Load the saved model and tokenizer
loaded_tokenizer = BertTokenizer.from_pretrained(model_path)
loaded_model = BertForSequenceClassification.from_pretrained(model_path)

# Move model to GPU if available
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
loaded_model.to(device)
loaded_model.eval()

print('Model and tokenizer loaded successfully.')

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Model and tokenizer loaded successfully.


In [ ]:
import numpy as np

def predict_spam(texts, batch_size=32):
    all_preds = []
    # Process the texts in batches to avoid CUDA Out of Memory errors
    for i in range(0, len(texts), batch_size):
        batch = texts[i : i + batch_size]

        # Tokenize batch
        inputs = loaded_tokenizer(
            batch,
            padding=True,
            truncation=True,
            max_length=512,
            return_tensors='pt'
        ).to(device)

        with torch.no_grad():
            outputs = loaded_model(**inputs)
            # Get the predicted class (0 or 1)
            batch_predictions = torch.argmax(outputs.logits, dim=-1)
            all_preds.extend(batch_predictions.cpu().numpy())

    return np.array(all_preds)

# Load the Enron dataset
df_enron = pd.read_csv('../DATASETS/enron_preprocessed.csv')

# Handle potential missing values in 'processed_text'
df_enron['processed_text'] = df_enron['processed_text'].fillna('')

texts = df_enron['processed_text'].tolist()
labels = df_enron['label'].tolist()

print(f'Processing {len(texts)} samples...')
results = predict_spam(texts)

print('Accuracy:', accuracy_score(labels, results))
print('F1 Score:', f1_score(labels, results))
print('Classification Report:')
print(classification_report(labels, results))

Processing 33716 samples...
Accuracy: 0.5206430181516194
F1 Score: 0.3126063286832256
Classification Report:
              precision    recall  f1-score   support

           0       0.51      0.84      0.63     16545
           1       0.58      0.21      0.31     17171

    accuracy                           0.52     33716
   macro avg       0.54      0.53      0.47     33716
weighted avg       0.54      0.52      0.47     33716



In [11]:
!pip install peft

In [12]:
from sklearn.utils import resample

df_few_shot = df_enron

fs_texts = df_few_shot['processed_text'].tolist()
fs_labels = df_few_shot['label'].tolist()

# Tokenize
fs_encodings = tokenizer(fs_texts, padding=True, truncation=True, max_length=512, return_tensors='pt')
fs_dataset = SpamDataset(fs_encodings, fs_labels)

print(f'Prepared {len(fs_dataset)} examples for LoRA fine-tuning.')

Prepared 33716 examples for LoRA fine-tuning.


In [ ]:
from peft import LoraConfig, get_peft_model, TaskType

# Define LoRA Config
lora_config = LoraConfig(
    task_type=TaskType.SEQ_CLS,
    r=8,
    lora_alpha=16,
    lora_dropout=0.1,
    target_modules=["query", "value"] # Target BERT's attention layers
)

# Load the model again to ensure a clean state or use the current 'loaded_model'
# We will use the model that was already fine-tuned on SpamAssassin
peft_model = get_peft_model(loaded_model, lora_config)
peft_model.print_trainable_parameters()

# Updated training arguments for few-shot
fs_training_args = TrainingArguments(
    output_dir='./lora_results',
    num_train_epochs=2, # Slightly more epochs for few-shot
    per_device_train_batch_size=4,
    learning_rate=2e-4, # LoRA typically needs a higher LR
    logging_steps=5,
    eval_strategy='no',
    save_strategy='no',
)

fs_trainer = Trainer(
    model=peft_model,
    args=fs_training_args,
    train_dataset=fs_dataset,
)

print('Starting LoRA training...')
fs_trainer.train()

trainable params: 296,450 || all params: 109,780,228 || trainable%: 0.2700
Starting LoRA training...


/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Step,Training Loss
5,4.632730
10,3.841813
15,2.260295
20,2.575299
25,2.361969
30,1.250178
35,1.311337
40,0.994970
45,1.217343
50,1.138289


TrainOutput(global_step=8430, training_loss=0.03222029058264501, metrics={'train_runtime': 3466.8708, 'train_samples_per_second': 19.45, 'train_steps_per_second': 2.432, 'total_flos': 1.780351462981632e+16, 'train_loss': 0.03222029058264501, 'epoch': 2.0})

In [14]:
# Evaluate again on the full Enron dataset (excluding the 200 used for training if you want strict validation,
# but here we'll check the overall improvement on the full 'texts' list from earlier).

def predict_peft(texts, batch_size=32):
    peft_model.eval()
    all_preds = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i : i + batch_size]
        inputs = loaded_tokenizer(batch, padding=True, truncation=True, max_length=512, return_tensors='pt').to(device)
        with torch.no_grad():
            outputs = peft_model(**inputs)
            batch_predictions = torch.argmax(outputs.logits, dim=-1)
            all_preds.extend(batch_predictions.cpu().numpy())
    return np.array(all_preds)

print('Evaluating LoRA model on Enron dataset...')
lora_results = predict_peft(texts)

print('New Accuracy:', accuracy_score(labels, lora_results))
print('New F1 Score:', f1_score(labels, lora_results))
print(classification_report(labels, lora_results))

Evaluating LoRA model on Enron dataset...
New Accuracy: 0.9996440858939376
New F1 Score: 0.9996506956977353
              precision    recall  f1-score   support

           0       1.00      1.00      1.00     16545
           1       1.00      1.00      1.00     17171

    accuracy                           1.00     33716
   macro avg       1.00      1.00      1.00     33716
weighted avg       1.00      1.00      1.00     33716



In [ ]:
# Convert the predictions tensor to a standard Python list
preds_list = preds.tolist()

# Collect the misclassified examples
misclassified_data = []
for text, true_label, pred_label in zip(val_texts, val_labels, preds_list):
    if true_label != pred_label:
        misclassified_data.append({
            'Text': text,
            'True Label': true_label,
            'Predicted Label': pred_label
        })

# Create a DataFrame for clean formatting in Colab
misclassified_df = pd.DataFrame(misclassified_data)

print(f"Total misclassified examples: {len(misclassified_df)} out of {len(val_labels)}\n")

# Optional: Set pandas to show the full text without truncating it
pd.set_option('display.max_colwidth', None)

# Display the dataframe (display() renders nicely in Colab)
display(misclassified_df)